In [15]:
import os
import re
import json
import glob
import hashlib
import warnings

import pandas as pd

warnings.filterwarnings("ignore")

DATA_DIR = "data"
OUTPUT_DIR = "processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [16]:
def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue  # skip malformed lines
    return records


jsonl_files = sorted(glob.glob(os.path.join(DATA_DIR, "*_pr_data.jsonl")))
print(f"Found {len(jsonl_files)} dataset files:")
for f in jsonl_files:
    print(" -", f)

all_prs = []
for path in jsonl_files:
    prs = load_jsonl(path)
    all_prs.extend(prs)
    print(f"  {os.path.basename(path)}: {len(prs)} PRs")

print(f"\nTotal PRs loaded: {len(all_prs)}")


Found 14 dataset files:
 - data\apache_kafka_pr_data.jsonl
 - data\django_django_pr_data.jsonl
 - data\fastapi_fastapi_pr_data.jsonl
 - data\gin-gonic_gin_pr_data.jsonl
 - data\kubernetes_kubernetes_pr_data.jsonl
 - data\microsoft_vscode_pr_data.jsonl
 - data\opencv_opencv_pr_data.jsonl
 - data\pallets_flask_pr_data.jsonl
 - data\pydantic_pydantic_pr_data.jsonl
 - data\pytorch_pytorch_pr_data.jsonl
 - data\react_react_pr_data.jsonl
 - data\spring-projects_spring-framework_pr_data.jsonl
 - data\tensorflow_tensorflow_pr_data.jsonl
 - data\vercel_next.js_pr_data.jsonl
  apache_kafka_pr_data.jsonl: 10 PRs
  django_django_pr_data.jsonl: 10 PRs
  fastapi_fastapi_pr_data.jsonl: 10 PRs
  gin-gonic_gin_pr_data.jsonl: 10 PRs
  kubernetes_kubernetes_pr_data.jsonl: 10 PRs
  microsoft_vscode_pr_data.jsonl: 10 PRs
  opencv_opencv_pr_data.jsonl: 10 PRs
  pallets_flask_pr_data.jsonl: 10 PRs
  pydantic_pydantic_pr_data.jsonl: 10 PRs
  pytorch_pytorch_pr_data.jsonl: 10 PRs
  react_react_pr_data.jsonl: 1

In [17]:
BOT_SUFFIXES = ("[bot]",)
BOT_NAMES = {"github-actions", "codecov", "dependabot", "coderabbitai", "sonarcloud"}

LOW_CONTENT_PATTERNS = [
    r"^lgtm[.! ]*$",
    r"^\+1$",
    r"^thanks?[.! ]*$",
    r"^ok(ay)?[.! ]*$",
    r"^done[.! ]*$",
    r"^:\+1:$",
    r"^👍+$",
    r"^\s*$",
]
LOW_CONTENT_RE = re.compile("|".join(LOW_CONTENT_PATTERNS), re.IGNORECASE)

QUOTE_LINE_RE = re.compile(r"^>.*$", re.MULTILINE)
HTML_COMMENT_RE = re.compile(r"<!--.*?-->", re.DOTALL)
MULTI_BLANK_RE = re.compile(r"\n{3,}")


def is_bot(username):
    if not username:
        return False
    lower = username.lower()
    return username.endswith(BOT_SUFFIXES) or lower in BOT_NAMES


def clean_text(text):
    if not text:
        return ""
    text = HTML_COMMENT_RE.sub("", text)
    text = QUOTE_LINE_RE.sub("", text)
    text = MULTI_BLANK_RE.sub("\n\n", text)
    return text.strip()


def is_low_content(text):
    if not text:
        return True
    stripped = text.strip()
    if len(stripped) < 4:
        return True
    return bool(LOW_CONTENT_RE.match(stripped))


In [18]:
def build_review_comment_rows(prs):
    rows = []
    for pr in prs:
        repo = pr.get("repository")
        pr_number = pr.get("pr_number")
        language = pr.get("repository_language")
        pr_title = pr.get("title") or ""
        pr_description = clean_text(pr.get("description")) or "(no description provided)"

        # map filename -> patch for this PR
        file_patches = {f["filename"]: f.get("patch") for f in pr.get("files", [])}

        for rc in pr.get("review_comments", []):
            user = rc.get("user")
            body = clean_text(rc.get("body"))

            if is_bot(user) or is_low_content(body):
                continue

            patch = file_patches.get(rc.get("path"))
            if not patch:
                continue

            rows.append({
                "repo": repo,
                "language": language,
                "pr_number": pr_number,
                "pr_title": pr_title,
                "pr_description": pr_description,
                "file": rc.get("path"),
                "line": rc.get("line"),
                "diff": patch,
                "comment": body,
                "user": user,
            })
    return rows


rows = build_review_comment_rows(all_prs)
df = pd.DataFrame(rows)
print(f"Built {len(df)} (diff, comment) pairs before dedup/filtering.")
df.head(3)


Built 667 (diff, comment) pairs before dedup/filtering.


,repo,language,pr_number,pr_title,pr_description,file,line,diff,comment,user
0,apache/kafka,Java,22977,KAFKA-18628: Deprecate broker.id config (KIP-1...,Implement [KIP-1232](https://cwiki.apache.org/...,storage/src/main/java/org/apache/kafka/server/...,NaN,"@@ -47,7 +47,6 @@\n import org.apache.kafka.se...",Can you please share why this isn't updated to...,gaurav-narula
1,apache/kafka,Java,22977,KAFKA-18628: Deprecate broker.id config (KIP-1...,Implement [KIP-1232](https://cwiki.apache.org/...,storage/src/main/java/org/apache/kafka/server/...,NaN,"@@ -47,7 +47,6 @@\n import org.apache.kafka.se...",Same here,gaurav-narula
2,apache/kafka,Java,22977,KAFKA-18628: Deprecate broker.id config (KIP-1...,Implement [KIP-1232](https://cwiki.apache.org/...,core/src/test/scala/unit/kafka/server/KafkaCon...,NaN,"@@ -45,6 +45,8 @@ import org.apache.logging.lo...",Please consider parameterising this against doLog,gaurav-narula


In [19]:
def format_input(row):
    return (
        f"Repository: {row['repo']}\n"
        f"Language: {row['language']}\n"
        f"PR Title: {row['pr_title']}\n"
        f"PR Description: {row['pr_description']}\n"
        f"File: {row['file']}\n"
        f"\nDiff:\n{row['diff']}"
    )


In [20]:
before = len(df)

# exact duplicate pairs
df = df.drop_duplicates(subset=["diff", "comment"])

# near-duplicate detection: same normalized comment text used > N times overall
def normalize(text):
    return re.sub(r"\s+", " ", text.strip().lower())

df["_norm_comment"] = df["comment"].apply(normalize)
counts = df["_norm_comment"].value_counts()
templated = set(counts[counts > 20].index)  # tune threshold as needed
df = df[~df["_norm_comment"].isin(templated)]
df = df.drop(columns=["_norm_comment"])

print(f"Rows before: {before}, after dedup: {len(df)} "
      f"(removed {before - len(df)})")


Rows before: 667, after dedup: 658 (removed 9)


In [21]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    def count_tokens(text):
        return len(enc.encode(text or ""))
except ImportError:
    print("tiktoken not installed, falling back to whitespace token count "
          "(install with: pip install tiktoken)")
    def count_tokens(text):
        return len((text or "").split())

# Measure the FULL formatted input (repo + language + title + description + diff),
# since that's what actually gets tokenized as the model input, not just the raw diff.
df["formatted_input"] = df.apply(format_input, axis=1)
df["input_tokens"] = df["formatted_input"].apply(count_tokens)
df["comment_tokens"] = df["comment"].apply(count_tokens)

print(df[["input_tokens", "comment_tokens"]].describe())


       input_tokens  comment_tokens
count    658.000000      658.000000
mean    3472.749240       49.876900
std     2983.407043       65.131795
min      123.000000        1.000000
25%     1078.750000       14.000000
50%     1915.000000       30.000000
75%     4552.000000       57.750000
max     9991.000000      741.000000


In [22]:
MAX_INPUT_TOKENS = 2000
MAX_COMMENT_TOKENS = 500
MIN_COMMENT_TOKENS = 3

before = len(df)
df = df[
    (df["input_tokens"] <= MAX_INPUT_TOKENS)
    & (df["comment_tokens"] <= MAX_COMMENT_TOKENS)
    & (df["comment_tokens"] >= MIN_COMMENT_TOKENS)
]
print(f"Rows before: {before}, after length filtering: {len(df)} "
      f"(removed {before - len(df)})")


Rows before: 658, after length filtering: 329 (removed 329)


In [23]:
from sklearn.model_selection import train_test_split

df["pr_key"] = df["repo"] + "#" + df["pr_number"].astype(str)
unique_prs = df["pr_key"].unique()

train_prs, temp_prs = train_test_split(unique_prs, test_size=0.2, random_state=42)
val_prs, test_prs = train_test_split(temp_prs, test_size=0.5, random_state=42)

train_df = df[df["pr_key"].isin(train_prs)]
val_df = df[df["pr_key"].isin(val_prs)]
test_df = df[df["pr_key"].isin(test_prs)]

print(f"Train: {len(train_df)} rows ({len(train_prs)} PRs)")
print(f"Val:   {len(val_df)} rows ({len(val_prs)} PRs)")
print(f"Test:  {len(test_df)} rows ({len(test_prs)} PRs)")


Train: 303 rows (23 PRs)
Val:   6 rows (3 PRs)
Test:  20 rows (3 PRs)


In [24]:
INSTRUCTION = (
    "You are an experienced software engineer performing a code review. "
    "Given the following pull request context and code diff, write a concise, helpful review comment."
)


def to_sft_records(split_df):
    records = []
    for _, row in split_df.iterrows():
        records.append({
            "instruction": INSTRUCTION,
            "input": row["formatted_input"],
            "output": row["comment"],
            "metadata": {
                "repo": row["repo"],
                "language": row["language"],
                "pr_number": int(row["pr_number"]) if pd.notna(row["pr_number"]) else None,
                "file": row["file"],
                "line": int(row["line"]) if pd.notna(row["line"]) else None,
            },
        })
    return records


def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} examples -> {path}")


save_jsonl(to_sft_records(train_df), os.path.join(OUTPUT_DIR, "train.jsonl"))
save_jsonl(to_sft_records(val_df), os.path.join(OUTPUT_DIR, "val.jsonl"))
save_jsonl(to_sft_records(test_df), os.path.join(OUTPUT_DIR, "test.jsonl"))


Saved 303 examples -> processed\train.jsonl
Saved 6 examples -> processed\val.jsonl
Saved 20 examples -> processed\test.jsonl


In [25]:
print("By repository:")
print(df["repo"].value_counts())
print("\nBy language:")
print(df["language"].value_counts())
print("\nTotal SFT examples:", len(train_df) + len(val_df) + len(test_df))


By repository:
repo
kubernetes/kubernetes    129
apache/kafka             117
django/django             20
microsoft/vscode          14
react/react               13
vercel/next.js            12
pytorch/pytorch           12
gin-gonic/gin             10
pydantic/pydantic          2
Name: count, dtype: int64

By language:
language
Go            139
Java          117
Python         34
JavaScript     25
TypeScript     14
Name: count, dtype: int64

Total SFT examples: 329
